In [1]:
# Enhanced PII Anonymizer - Plug and Play Version
from typing import Dict, List, Optional, Any, Union, Tuple
import langdetect
from dataclasses import dataclass, asdict
import json
import hashlib
import secrets
from abc import ABC, abstractmethod
from enum import Enum
import logging

# Core Types (Enhanced)
class Language(Enum):
    ENGLISH = "en"
    GERMAN = "de"
    AUTO = "auto"  # New: Auto-detect language

@dataclass
class SessionEntity:
    """Represents a stored entity for session continuity"""
    entity_id: str
    original_value: str
    fake_value: str
    entity_type: str
    language: str
    first_seen: str  # timestamp
    last_used: str   # timestamp
    usage_count: int = 1

@dataclass
class AnonymizationConfig:
    """Simplified configuration for plug-and-play usage"""
    confidence_threshold: float = 0.65
    preserve_format: bool = True
    store_entities: bool = True
    session_id: Optional[str] = None
    custom_entities: Optional[List[str]] = None

@dataclass
class AnonymizationResult:
    """Result with session information"""
    anonymized_text: str
    detected_entities: List[Dict[str, Any]]
    detected_language: str
    session_id: str
    metadata: Dict[str, Any]

class EntityStorage:
    """Manages persistent entity storage for session continuity"""
    
    def __init__(self, storage_backend: str = "memory"):
        self.storage_backend = storage_backend
        self._memory_store: Dict[str, Dict[str, SessionEntity]] = {}
    
    def store_entity(self, session_id: str, entity: SessionEntity) -> None:
        """Store entity for future use"""
        if session_id not in self._memory_store:
            self._memory_store[session_id] = {}
        
        # Check if we already have this original value
        existing_entity = self._find_existing_entity(session_id, entity.original_value)
        if existing_entity:
            existing_entity.usage_count += 1
            existing_entity.last_used = entity.last_used
        else:
            self._memory_store[session_id][entity.entity_id] = entity
    
    def get_entity_by_original(self, session_id: str, original_value: str) -> Optional[SessionEntity]:
        """Get existing entity by original value"""
        return self._find_existing_entity(session_id, original_value)
    
    def get_entity_by_fake(self, session_id: str, fake_value: str) -> Optional[SessionEntity]:
        """Get entity by fake value for deanonymization"""
        if session_id not in self._memory_store:
            return None
        
        for entity in self._memory_store[session_id].values():
            if entity.fake_value == fake_value:
                return entity
        return None
    
    def get_session_entities(self, session_id: str) -> Dict[str, SessionEntity]:
        """Get all entities for a session"""
        return self._memory_store.get(session_id, {})
    
    def _find_existing_entity(self, session_id: str, original_value: str) -> Optional[SessionEntity]:
        """Find existing entity by original value"""
        if session_id not in self._memory_store:
            return None
        
        for entity in self._memory_store[session_id].values():
            if entity.original_value == original_value:
                return entity
        return None

class LanguageDetector:
    """Enhanced language detection with fallback"""
    
    @staticmethod
    def detect_language(text: str) -> str:
        """Detect language with confidence scoring"""
        if not text or len(text.strip()) < 10:
            return "en"  # Default fallback
        
        try:
            detected = langdetect.detect(text)
            # Map common language codes
            language_mapping = {
                'en': 'en',
                'de': 'de',
                'fr': 'en',  # Fallback to English for unsupported languages
                'es': 'en',
                'it': 'en'
            }
            return language_mapping.get(detected, 'en')
        except:
            # Fallback detection using simple heuristics
            german_indicators = ['der', 'die', 'das', 'und', 'ist', 'mit', 'für', 'auf', 'eine', 'einen']
            english_indicators = ['the', 'and', 'is', 'with', 'for', 'on', 'a', 'an', 'this', 'that']
            
            text_lower = text.lower()
            german_count = sum(1 for word in german_indicators if word in text_lower)
            english_count = sum(1 for word in english_indicators if word in text_lower)
            
            return 'de' if german_count > english_count else 'en'

class PIIAnonymizer:
    """Main plug-and-play anonymizer class"""
    
    def __init__(self, storage_backend: str = "memory"):
        self.entity_storage = EntityStorage(storage_backend)
        self.language_detector = LanguageDetector()
        self._setup_presidio_engines()
    
    def _setup_presidio_engines(self):
        """Setup Presidio engines for both languages"""
        from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
        from presidio_anonymizer import AnonymizerEngine
        from presidio_anonymizer.entities import InvalidParamException, OperatorConfig
        from presidio_anonymizer.operators import Operator, OperatorType
        
        # Create enhanced analyzers with your existing recognizers
        self.analyzers = {
            'en': self._create_english_analyzer(),
            'de': self._create_german_analyzer()
        }
        
        # Create anonymizer with custom operators
        self.anonymizer = AnonymizerEngine()
        self._register_custom_operators()
    
    def anonymize(
        self, 
        text: str, 
        config: Optional[AnonymizationConfig] = None
    ) -> AnonymizationResult:
        """
        Main anonymization method using Presidio engines
        """
        if config is None:
            config = AnonymizationConfig()
        
        # Generate session ID if not provided
        session_id = config.session_id or self._generate_session_id(text)
        
        # Detect language
        detected_language = self.language_detector.detect_language(text)
        
        # Get analyzer for detected language
        analyzer = self.analyzers[detected_language]
        
        # Analyze text using Presidio
        entities_to_check = config.custom_entities or self._get_default_entities(detected_language)
        analysis_results = analyzer.analyze(
            text=text,
            entities=entities_to_check,
            language=detected_language
        )
        
        # Filter by confidence
        filtered_results = [
            result for result in analysis_results 
            if result.score >= config.confidence_threshold
        ]
        
        if not filtered_results:
            # No entities found, return original text
            return AnonymizationResult(
                anonymized_text=text,
                detected_entities=[],
                detected_language=detected_language,
                session_id=session_id,
                metadata={'total_entities': 0, 'entity_types': []}
            )
        
        # Check for existing entities in session and prepare anonymization operators
        operator_configs = {}
        detected_entities = []
        
        for result in filtered_results:
            original_value = text[result.start:result.end]
            entity_type = result.entity_type
            
            # Check if we already have this entity stored
            existing_entity = self.entity_storage.get_entity_by_original(session_id, original_value)
            
            if existing_entity:
                # Use existing fake value
                fake_value = existing_entity.fake_value
                existing_entity.usage_count += 1
                existing_entity.last_used = self._get_timestamp()
            else:
                # Generate new fake value using our custom operator
                custom_operator = self.custom_operators[detected_language]
                fake_value = custom_operator.operate(
                    original_value, 
                    {'entity_type': entity_type}
                )
                
                if config.store_entities:
                    # Store new entity
                    new_entity = SessionEntity(
                        entity_id=self._generate_entity_id(entity_type, original_value),
                        original_value=original_value,
                        fake_value=fake_value,
                        entity_type=entity_type,
                        language=detected_language,
                        first_seen=self._get_timestamp(),
                        last_used=self._get_timestamp()
                    )
                    self.entity_storage.store_entity(session_id, new_entity)
            
            # Configure Presidio to use our custom replacement
            operator_configs[entity_type] = OperatorConfig(
                "replace", 
                {"new_value": fake_value}
            )
            
            detected_entities.append({
                'entity_type': entity_type,
                'original_value': original_value,
                'fake_value': fake_value,
                'start': result.start,
                'end': result.end,
                'confidence': result.score
            })
        
        # Use Presidio anonymizer with our custom replacements
        from presidio_anonymizer.entities import RecognizerResult, OperatorConfig
        
        # Convert analysis results to anonymizer format
        presidio_results = []
        for result in filtered_results:
            original_value = text[result.start:result.end]
            entity_type = result.entity_type
            
            # Find the fake value for this entity
            fake_value = next(
                (e['fake_value'] for e in detected_entities 
                 if e['original_value'] == original_value and e['entity_type'] == entity_type),
                f"[REDACTED_{entity_type}]"
            )
            
            presidio_results.append(
                RecognizerResult(
                    entity_type=entity_type,
                    start=result.start,
                    end=result.end,
                    score=result.score
                )
            )
        
        # Anonymize using Presidio with our custom operators
        anonymized_result = self.anonymizer.anonymize(
            text=text,
            analyzer_results=presidio_results,
            operators=operator_configs
        )
        
        return AnonymizationResult(
            anonymized_text=anonymized_result.text,
            detected_entities=detected_entities,
            detected_language=detected_language,
            session_id=session_id,
            metadata={
                'total_entities': len(detected_entities),
                'entity_types': list(set(e['entity_type'] for e in detected_entities)),
                'confidence_threshold': config.confidence_threshold,
                'presidio_version': True
            }
        )
    
    def deanonymize(self, anonymized_text: str, session_id: str) -> str:
        """
        Deanonymize text using stored entities
        """
        session_entities = self.entity_storage.get_session_entities(session_id)
        
        # Sort entities by fake value length (longest first) to avoid partial replacements
        sorted_entities = sorted(
            session_entities.values(),
            key=lambda x: len(x.fake_value),
            reverse=True
        )
        
        deanonymized_text = anonymized_text
        for entity in sorted_entities:
            if entity.fake_value in deanonymized_text:
                deanonymized_text = deanonymized_text.replace(
                    entity.fake_value, 
                    entity.original_value
                )
        
        return deanonymized_text
    
    def get_session_stats(self, session_id: str) -> Dict[str, Any]:
        """Get statistics for a session"""
        entities = self.entity_storage.get_session_entities(session_id)
        
        if not entities:
            return {'total_entities': 0, 'entity_types': {}}
        
        entity_types = {}
        for entity in entities.values():
            if entity.entity_type not in entity_types:
                entity_types[entity.entity_type] = {
                    'count': 0,
                    'total_usage': 0
                }
            entity_types[entity.entity_type]['count'] += 1
            entity_types[entity.entity_type]['total_usage'] += entity.usage_count
        
        return {
            'total_entities': len(entities),
            'entity_types': entity_types,
            'session_id': session_id
        }
    
    # Helper methods
    def _generate_session_id(self, text: str) -> str:
        """Generate session ID based on text hash"""
        return hashlib.md5(text.encode()).hexdigest()[:12]
    
    def _generate_entity_id(self, entity_type: str, original_value: str) -> str:
        """Generate unique entity ID"""
        hash_input = f"{entity_type}:{original_value}"
        return f"{entity_type}_{hashlib.sha256(hash_input.encode()).hexdigest()[:8]}"
    
    def _generate_fake_value(self, entity_type: str, original_value: str, language: str) -> str:
        """DEPRECATED - Now using Presidio's custom operators"""
        # This method is kept for backward compatibility but should not be used
        # The actual fake value generation is now handled by Presidio's custom operators
        pass
    
    def _create_english_analyzer(self) -> 'AnalyzerEngine':
        """Create English analyzer with your existing custom recognizers"""
        from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
        from presidio_analyzer.nlp_engine import NlpEngineProvider
        
        # Use your existing recognizers from the original code
        from recognizers.english_recognizers import EnglishRecognizers
        
        registry = RecognizerRegistry()
        registry.load_predefined_recognizers(languages=["en"])
        
        # Add your custom English recognizers
        english_recognizers = EnglishRecognizers()
        for recognizer in english_recognizers.get_recognizers():
            registry.add_recognizer(recognizer)
        
        # Setup NLP engine
        nlp_config = {
            "nlp_engine_name": "spacy",
            "models": [{"lang_code": "en", "model_name": "en_core_web_lg"}],
        }
        provider = NlpEngineProvider(nlp_configuration=nlp_config)
        nlp_engine = provider.create_engine()
        
        return AnalyzerEngine(registry=registry, nlp_engine=nlp_engine)
    
    def _create_german_analyzer(self) -> 'AnalyzerEngine':
        """Create German analyzer with your existing custom recognizers"""
        from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
        from presidio_analyzer.nlp_engine import NlpEngineProvider
        
        # Use your existing recognizers from the original code
        from recognizers.german_recognizers import GermanRecognizers
        from recognizers.english_recognizers import EnglishRecognizers
        
        registry = RecognizerRegistry()
        registry.load_predefined_recognizers(languages=["de"])
        
        # Add your custom German recognizers
        german_recognizers = GermanRecognizers()
        for recognizer in german_recognizers.get_recognizers():
            registry.add_recognizer(recognizer)
        
        # Add some English recognizers that work for German too
        english_recognizers = EnglishRecognizers()
        for recognizer in english_recognizers.get_recognizers():
            if recognizer.supported_entities[0] in ["CRYPTO_WALLET", "MEDICAL_LICENSE"]:
                registry.add_recognizer(recognizer)
        
        # Setup NLP engine
        nlp_config = {
            "nlp_engine_name": "spacy",
            "models": [{"lang_code": "de", "model_name": "de_core_news_lg"}],
        }
        provider = NlpEngineProvider(nlp_configuration=nlp_config)
        nlp_engine = provider.create_engine()
        
        return AnalyzerEngine(
            registry=registry,
            nlp_engine=nlp_engine,
            default_score_threshold=0.65
        )
    
    def _register_custom_operators(self):
        """Register custom anonymization operators with Presidio"""
        from presidio_anonymizer.operators import Operator, OperatorType
        from presidio_anonymizer.entities import OperatorConfig
        
        # Create custom operators that generate realistic fake data
        class RealisticReplaceOperator(Operator):
            """Custom operator that replaces with realistic fake values"""
            
            def __init__(self, language: str = "en"):
                self.language = language
            
            def operate(self, text: str, params: dict = None) -> str:
                entity_type = params.get('entity_type', 'UNKNOWN')
                return self._generate_realistic_fake_value(entity_type, text)
            
            def validate(self, params: dict = None) -> None:
                pass
            
            def operator_name(self) -> str:
                return "realistic_replace"
            
            def operator_type(self) -> OperatorType:
                return OperatorType.Anonymize
            
            def _generate_realistic_fake_value(self, entity_type: str, original_value: str) -> str:
                """Generate realistic fake values using our enhanced logic"""
                # Use the same logic as before but now properly integrated with Presidio
                if self.language == 'de':
                    first_names = ['Max', 'Anna', 'Thomas', 'Sarah', 'Michael', 'Lisa']
                    last_names = ['Mustermann', 'Schmidt', 'Müller', 'Weber', 'Fischer']
                    cities = ['Berlin', 'München', 'Hamburg', 'Frankfurt', 'Stuttgart']
                    streets = ['Hauptstraße', 'Schulstraße', 'Bahnhofstraße', 'Kirchstraße']
                else:
                    first_names = ['John', 'Jane', 'Michael', 'Sarah', 'David', 'Emily']
                    last_names = ['Smith', 'Johnson', 'Brown', 'Davis', 'Wilson']
                    cities = ['Springfield', 'Franklin', 'Riverside', 'Georgetown']
                    streets = ['Main St', 'Oak St', 'Pine St', 'Maple Ave']
                
                generators = {
                    'PERSON': lambda: f"{secrets.choice(first_names)} {secrets.choice(last_names)}",
                    'EMAIL_ADDRESS': lambda: f"{secrets.choice(first_names).lower()}.{secrets.choice(last_names).lower()}@example-corp.com",
                    'PHONE_NUMBER': lambda: self._generate_phone_number(),
                    'LOCATION': lambda: secrets.choice(cities),
                    'CREDIT_CARD': lambda: self._generate_credit_card(),
                    'IBAN_CODE': lambda: self._generate_iban(),
                    'IP_ADDRESS': lambda: f"10.0.{secrets.randbelow(255)}.{secrets.randbelow(255)}",
                    'URL': lambda: f"https://example-demo.com/{secrets.token_hex(3)}",
                }
                
                # Add German-specific generators
                if self.language == 'de':
                    german_generators = {
                        'DE_TAX_ID': lambda: f"12 345 678 {secrets.randbelow(900)+100:03d}",
                        'DE_PHONE_NUMBER': lambda: f"+49 {secrets.randbelow(900)+100} {secrets.randbelow(90000000)+10000000:08d}",
                        'DE_IBAN': lambda: f"DE89 3704 0044 0532 0130 {secrets.randbelow(90)+10:02d}",
                        'DE_STREET_ADDRESS': lambda: f"{secrets.choice(streets)} {secrets.randbelow(200)+1}",
                        'DE_POSTAL_CODE': lambda: f"{secrets.randbelow(90000)+10000:05d}",
                        'DE_ID_CARD': lambda: f"T{secrets.randbelow(90000000)+10000000:08d}",
                        'DE_PASSPORT': lambda: f"C{secrets.choice(['F', 'P'])}{secrets.randbelow(9000000)+1000000:07d}",
                        'DE_VAT_ID': lambda: f"DE{secrets.randbelow(900000000)+100000000:09d}",
                    }
                    generators.update(german_generators)
                
                generator = generators.get(entity_type)
                if generator:
                    return generator()
                else:
                    return f"[REDACTED_{entity_type}_{secrets.randbelow(999):03d}]"
            
            def _generate_phone_number(self) -> str:
                if self.language == 'de':
                    return f"+49 {secrets.randbelow(900)+100} {secrets.randbelow(90000000)+10000000:08d}"
                else:
                    return f"+1 {secrets.randbelow(900)+100} {secrets.randbelow(900)+100}-{secrets.randbelow(9000)+1000:04d}"
            
            def _generate_credit_card(self) -> str:
                prefixes = ['4000', '5555', '3714', '6011']  # Test prefixes
                prefix = secrets.choice(prefixes)
                if prefix == '3714':  # Amex
                    return f"{prefix} {secrets.randbelow(900000)+100000:06d} {secrets.randbelow(90000)+10000:05d}"
                else:
                    return f"{prefix} {secrets.randbelow(9000)+1000:04d} {secrets.randbelow(9000)+1000:04d} {secrets.randbelow(9000)+1000:04d}"
            
            def _generate_iban(self) -> str:
                if self.language == 'de':
                    return f"DE89 3704 0044 0532 0130 {secrets.randbelow(90)+10:02d}"
                else:
                    return f"GB29 NWBK 6016 1331 9268 {secrets.randbelow(90)+10:02d}"
        
        # Store operators for both languages
        self.custom_operators = {
            'en': RealisticReplaceOperator('en'),
            'de': RealisticReplaceOperator('de')
        }
    
    def _get_default_entities(self, language: str) -> List[str]:
        """Get default entities to check for each language"""
        base_entities = [
            "CREDIT_CARD", "EMAIL_ADDRESS", "IBAN_CODE", "IP_ADDRESS",
            "LOCATION", "PERSON", "PHONE_NUMBER", "URL", "DATE_TIME"
        ]
        
        if language == 'de':
            german_entities = [
                "DE_TAX_ID", "DE_PHONE_NUMBER", "DE_IBAN", 
                "DE_STREET_ADDRESS", "DE_PASSPORT", "DE_ID_CARD"
            ]
            return base_entities + german_entities
        
        return base_entities
    
    def _get_timestamp(self) -> str:
        """Get current timestamp"""
        from datetime import datetime
        return datetime.now().isoformat()


# Simple Usage Example
class LLMAnonymizer:
    """Wrapper class for LLM integration"""
    
    def __init__(self):
        self.anonymizer = PIIAnonymizer()
        self.current_session = None
    
    def process_user_input(self, user_text: str, session_id: str = None) -> Tuple[str, str]:
        """
        Process user input before sending to LLM
        Returns: (anonymized_text, session_id)
        """
        config = AnonymizationConfig(session_id=session_id)
        result = self.anonymizer.anonymize(user_text, config)
        self.current_session = result.session_id
        return result.anonymized_text, result.session_id
    
    def process_llm_response(self, llm_response: str, session_id: str) -> str:
        """
        Process LLM response to restore original entities
        Returns: deanonymized_response
        """
        return self.anonymizer.deanonymize(llm_response, session_id)
    
    def get_session_info(self, session_id: str = None) -> Dict[str, Any]:
        """Get information about current session"""
        sid = session_id or self.current_session
        return self.anonymizer.get_session_stats(sid) if sid else {}



In [2]:
# Usage Example:
if __name__ == "__main__":
    # Simple plug-and-play usage
    llm_anonymizer = LLMAnonymizer()
    
    # Process user input
    user_input = "Hi, I'm John Doe, my email is john@example.com and I live at 123 Main St."
    anonymized_input, session_id = llm_anonymizer.process_user_input(user_input)
    print(f"Anonymized: {anonymized_input}")
    
    # Simulate LLM response
    llm_response = f"Hello Person_abc, I've sent information to user123@example.com regarding City_xy."
    
    # Deanonymize response
    final_response = llm_anonymizer.process_llm_response(llm_response, session_id)
    print(f"Final response: {final_response}")
    
    # Get session stats
    stats = llm_anonymizer.get_session_info(session_id)
    print(f"Session stats: {stats}")

ImportError: cannot import name 'InvalidParamException' from 'presidio_anonymizer.entities' (c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\presidio_anonymizer\entities\__init__.py)